# Imports

In [21]:
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import deque

RESULTS_DIR = "Results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Packet class

In [22]:
class Packet:
    def __init__(self, pid, arrival_time, src_input, dest_output):
        self.pid = pid
        self.arrival_time = arrival_time
        self.src_input = src_input
        self.dest_output = dest_output

# Switch Classes

## FIFO switch class

In [23]:
class FIFOSwitch:
    def __init__(self, num_ports=3):
        self.num_ports            = num_ports
        self.queues               = [deque() for _ in range(num_ports)]
        self.time                 = 0
        self.sent_packets_log     = []
        self.hold_blocking_events = []
        self.backlog_history      = []   # entry t = packets remaining AFTER slot t sends
        self.step_logs            = []

    def add_packet(self, packet):
        self.queues[packet.src_input].append(packet)

    def get_backlog(self):
        return sum(len(q) for q in self.queues)

    def get_queue_state_string(self):
        state = []
        for i in range(self.num_ports):
            if self.queues[i]:
                qs = ' | '.join(f"{p.pid}(→O{p.dest_output})" for p in self.queues[i])
                state.append(f"I{i}: [{qs}]")
            else:
                state.append(f"I{i}: [empty]")
        return '\n        '.join(state)

    def step(self, record_log=True):
        # Arbitration: selected[output] = winning input port, or -1
        selected         = [-1] * self.num_ports
        selected_outputs = set()
        selected_inputs  = set()

        for i in range(self.num_ports):
            if self.queues[i] and i not in selected_inputs:
                head = self.queues[i][0]
                dest = head.dest_output
                if dest not in selected_outputs:
                    selected_inputs.add(i)
                    selected_outputs.add(dest)
                    selected[dest] = i

        original_heads = {
            i: self.queues[i][0]
            for i in range(self.num_ports) if self.queues[i]
        }

        # Send packets
        sent_this_slot      = []
        sent_packet_ids     = []
        already_sent_inputs = set()

        for output in range(self.num_ports):
            input_idx = selected[output]
            if input_idx != -1 and input_idx not in already_sent_inputs:
                packet = self.queues[input_idx].popleft()
                already_sent_inputs.add(input_idx)
                sent_this_slot.append(packet)
                sent_packet_ids.append(packet.pid)
                self.sent_packets_log.append({
                    'time':      self.time,
                    'packet_id': packet.pid,
                    'src':       packet.src_input,
                    'dest':      packet.dest_output
                })

        # HoL blocking detection
        hol_events_this_slot = []

        for i in range(self.num_ports):
            if i not in original_heads:
                continue
            head_packet = original_heads[i]
            for position, packet in enumerate(self.queues[i]):
                dest = packet.dest_output
                if position == 0:
                    continue
                if dest in selected_outputs:
                    continue
                if head_packet.dest_output != dest:
                    event = {
                        'time':      self.time,
                        'packet_id': packet.pid,
                        'src':       packet.src_input,
                        'dest':      dest,
                        'position':  position,
                        'reason':    (f"TRUE HoL: {packet.pid} wants O{dest} (idle) "
                                      f"but blocked by head {head_packet.pid} → O{head_packet.dest_output}"),
                        'type':      'TRUE_HOL'
                    }
                    self.hold_blocking_events.append(event)
                    hol_events_this_slot.append(event)
                    break

        # Arbitration-loss events
        for i in range(self.num_ports):
            if i not in original_heads:
                continue
            head = original_heads[i]
            if head.pid not in sent_packet_ids:
                dest = head.dest_output
                if dest in selected_outputs:
                    winning_input = selected[dest]
                    event = {
                        'time':      self.time,
                        'packet_id': head.pid,
                        'src':       head.src_input,
                        'dest':      dest,
                        'reason':    f"Lost arbitration to I{winning_input} for output O{dest}",
                        'type':      'ARBITRATION_LOSS'
                    }
                    self.hold_blocking_events.append(event)
                    hol_events_this_slot.append(event)

        # Record backlog AFTER sending — packets remaining after this slot
        self.backlog_history.append(self.get_backlog())

        if record_log:
            self.step_logs.append({
                'time':         self.time,
                'queue_state':  self.get_queue_state_string(),
                'sent_packets': [p.pid for p in sent_this_slot],
                'hol_events':   hol_events_this_slot
            })

        self.time += 1
        return sent_this_slot

    def record_idle(self):
        """Called when backlog is already 0 — no sending, just advance time."""
        self.backlog_history.append(0)
        self.time += 1

## Optimal VOQ Switch Class

In [24]:
class OptimalVOQSwitch:
    def __init__(self, num_ports=3):
        self.num_ports       = num_ports
        self.queues          = [[deque() for _ in range(num_ports)] for _ in range(num_ports)]
        self.time            = 0
        self.matchings       = []
        self.backlog_history = []
        self.step_logs       = []

    def add_packet(self, packet):
        self.queues[packet.src_input][packet.dest_output].append(packet)

    def get_backlog(self):
        return sum(len(self.queues[i][j])
                   for i in range(self.num_ports)
                   for j in range(self.num_ports))

    def get_queue_state_string(self):
        state = []
        for i in range(self.num_ports):
            non_empty = [f"O{j}:{len(self.queues[i][j])}"
                         for j in range(self.num_ports) if self.queues[i][j]]
            if non_empty:
                state.append(f"I{i}: [{' | '.join(non_empty)}]")
            else:
                state.append(f"I{i}: [empty]")
        return '\n        '.join(state)

    def get_all_matchings(self):
        matchings   = []
        output_used = [False] * self.num_ports

        def backtrack(inp, current):
            if inp == self.num_ports:
                matchings.append(list(current))
                return
            backtrack(inp + 1, current + [-1])
            for out in range(self.num_ports):
                if not output_used[out] and self.queues[inp][out]:
                    output_used[out] = True
                    backtrack(inp + 1, current + [out])
                    output_used[out] = False

        backtrack(0, [])
        return matchings

    def step_optimal(self, record_log=True):
        matchings = self.get_all_matchings()
        if not matchings:
            self.backlog_history.append(self.get_backlog())
            self.time += 1
            return []

        best = max(matchings, key=lambda m: sum(1 for d in m if d != -1))

        sent_packets = []
        for inp, dest in enumerate(best):
            if dest != -1:
                packet = self.queues[inp][dest].popleft()
                sent_packets.append(packet)

        # Record backlog AFTER sending
        self.backlog_history.append(self.get_backlog())

        if record_log:
            matching_str = [f"I{i}→O{d}" for i, d in enumerate(best) if d != -1]
            self.step_logs.append({
                'time':         self.time,
                'queue_state':  self.get_queue_state_string(),
                'matching':     matching_str,
                'sent_packets': [p.pid for p in sent_packets],
                'packets_sent': len(sent_packets)
            })

        self.matchings.append({
            'time':         self.time,
            'matching':     best,
            'sent_packets': [p.pid for p in sent_packets],
            'packets_sent': len(sent_packets)
        })

        self.time += 1
        return sent_packets

    def record_idle(self):
        self.backlog_history.append(0)
        self.time += 1

## iSLIP Switch Class

In [25]:
class iSLIPSwitch:
    def __init__(self, num_ports=3):
        self.num_ports        = num_ports
        self.queues           = [[deque() for _ in range(num_ports)] for _ in range(num_ports)]
        self.grant_pointer    = [0] * num_ports
        self.accept_pointer   = [0] * num_ports
        self.time             = 0
        self.sent_packets_log = []
        self.backlog_history  = []
        self.matchings        = []
        self.step_logs        = []

    def add_packet(self, packet):
        self.queues[packet.src_input][packet.dest_output].append(packet)

    def get_backlog(self):
        return sum(len(self.queues[i][j])
                   for i in range(self.num_ports)
                   for j in range(self.num_ports))

    def get_queue_state_string(self):
        state = []
        for i in range(self.num_ports):
            non_empty = [f"O{j}:{len(self.queues[i][j])}"
                         for j in range(self.num_ports) if self.queues[i][j]]
            if non_empty:
                state.append(f"I{i}: [{' | '.join(non_empty)}]")
            else:
                state.append(f"I{i}: [empty]")
        return '\n        '.join(state)

    def get_requests(self):
        return [[bool(self.queues[i][j]) for j in range(self.num_ports)]
                for i in range(self.num_ports)]

    def step_islip(self, record_log=True):
        requests       = self.get_requests()
        request_matrix = [f"I{i}→O{j}"
                          for i in range(self.num_ports)
                          for j in range(self.num_ports)
                          if requests[i][j]]

        matched_inputs    = set()
        matched_outputs   = set()
        final_matches     = {}
        iteration_details = []

        for iteration in range(self.num_ports):
            if len(matched_inputs) == self.num_ports or len(matched_outputs) == self.num_ports:
                break

            # GRANT phase
            grants        = {}
            grant_details = []

            for output in range(self.num_ports):
                if output in matched_outputs:
                    continue
                requesters = [i for i in range(self.num_ports)
                              if requests[i][output] and i not in matched_inputs]
                if requesters:
                    ptr = self.grant_pointer[output]
                    for offset in range(self.num_ports):
                        candidate = (ptr + offset) % self.num_ports
                        if candidate in requesters:
                            grants[output] = candidate
                            grant_details.append(
                                f"O{output} grants I{candidate} (pointer={ptr})")
                            break
                else:
                    grant_details.append(f"O{output}: no requests")

            # ACCEPT phase
            accepts        = {}
            accept_details = []

            for inp in range(self.num_ports):
                if inp in matched_inputs:
                    continue
                granted_outputs = [o for o, g in grants.items()
                                   if g == inp and o not in matched_outputs]
                if granted_outputs:
                    ptr = self.accept_pointer[inp]
                    for offset in range(self.num_ports):
                        candidate = (ptr + offset) % self.num_ports
                        if candidate in granted_outputs:
                            accepts[inp] = candidate
                            accept_details.append(
                                f"I{inp} accepts O{candidate} (pointer={ptr})")
                            break
                else:
                    accept_details.append(f"I{inp}: no grants")

            # UPDATE matches & pointers
            if accepts:
                for inp, out in accepts.items():
                    matched_inputs.add(inp)
                    matched_outputs.add(out)
                    final_matches[inp] = out

                    old_gp = self.grant_pointer[out]
                    old_ap = self.accept_pointer[inp]

                    self.grant_pointer[out]  = (inp + 1) % self.num_ports
                    self.accept_pointer[inp] = (out + 1) % self.num_ports

                    iteration_details.append({
                        'iteration':      iteration + 1,
                        'grant_details':  grant_details[:],
                        'accept_details': accept_details[:],
                        'matched':        f"I{inp}→O{out}",
                        'ptr_updates':    (f"Grant ptr O{out}: {old_gp}→{self.grant_pointer[out]}, "
                                           f"Accept ptr I{inp}: {old_ap}→{self.accept_pointer[inp]}")
                    })
            else:
                if grant_details or accept_details:
                    iteration_details.append({
                        'iteration':      iteration + 1,
                        'grant_details':  grant_details[:],
                        'accept_details': accept_details[:],
                        'matched':        "No matches this iteration",
                        'ptr_updates':    "No pointer updates"
                    })
                break

        # Send matched packets
        sent_packets = []
        for inp, out in final_matches.items():
            if self.queues[inp][out]:
                packet = self.queues[inp][out].popleft()
                sent_packets.append(packet)
                self.sent_packets_log.append({
                    'time':      self.time,
                    'packet_id': packet.pid,
                    'src':       inp,
                    'dest':      out
                })

        # Record backlog AFTER sending
        self.backlog_history.append(self.get_backlog())

        if record_log:
            self.step_logs.append({
                'time':              self.time,
                'queue_state':       self.get_queue_state_string(),
                'requests':          request_matrix,
                'iteration_details': iteration_details,
                'final_matches':     [f"I{i}→O{o}" for i, o in final_matches.items()],
                'sent_packets':      [p.pid for p in sent_packets],
                'grant_pointers':    self.grant_pointer[:],
                'accept_pointers':   self.accept_pointer[:]
            })

        self.matchings.append({
            'time':            self.time,
            'matching':        final_matches,
            'sent_packets':    [p.pid for p in sent_packets],
            'packets_sent':    len(sent_packets),
            'grant_pointers':  self.grant_pointer[:],
            'accept_pointers': self.accept_pointer[:]
        })

        self.time += 1
        return sent_packets

    def record_idle(self):
        self.backlog_history.append(0)
        self.time += 1

# Packet Loader

In [26]:
def load_packets():
    data = [
        ('p1',  0, 0, 0), ('p2',  0, 0, 1), ('p3',  0, 1, 0), ('p4',  0, 1, 2),
        ('p5',  0, 2, 0), ('p6',  1, 0, 2), ('p7',  1, 2, 1), ('p8',  2, 1, 1),
        ('p9',  2, 2, 2), ('p10', 3, 0, 1), ('p11', 3, 1, 0), ('p12', 3, 2, 1),
        ('p13', 4, 0, 0), ('p14', 4, 1, 2), ('p15', 4, 2, 2), ('p16', 5, 0, 2),
        ('p17', 5, 1, 1), ('p18', 5, 2, 0),
    ]
    return [Packet(pid, arr, src, dst) for pid, arr, src, dst in data]


# Simulation Runners

## FIFO Runner

In [27]:
def run_fifo_with_logging():
    packets = load_packets()
    fifo    = FIFOSwitch(3)

    print("\n" + "=" * 70)
    print("PART 1: FIFO STEP-BY-STEP SIMULATION")
    print("=" * 70)

    max_arr = max(p.arrival_time for p in packets)

    def do_step():
        print(f"\n{'=' * 60}")
        print(f"TIME SLOT t = {fifo.time}")
        print(f"{'=' * 60}")
        print("\nQueue BEFORE sending:")
        print(fifo.get_queue_state_string())

        fifo.step(record_log=True)
        last = fifo.step_logs[-1]

        print("\nPackets SENT:")
        if last['sent_packets']:
            for pid in last['sent_packets']:
                print(f"  → {pid}")
        else:
            print("  None")

        print("\nHoL / Arbitration Events:")
        if last['hol_events']:
            for e in last['hol_events']:
                print(f"  [{e['type']}] {e['packet_id']}: {e['reason']}")
        else:
            print("  None")

        print(f"\nBacklog remaining after slot {last['time']}: {fifo.backlog_history[-1]}")
        print("\nQueue AFTER sending:")
        print(fifo.get_queue_state_string())

    for t in range(max_arr + 1):
        for p in packets:
            if p.arrival_time == t:
                fifo.add_packet(p)
                print(f"\n[ARRIVAL] t={t}: {p.pid} (I{p.src_input} → O{p.dest_output})")
        if fifo.get_backlog() > 0:
            do_step()
        else:
            print(f"\n[IDLE] t={fifo.time}")
            fifo.record_idle()

    while fifo.get_backlog() > 0:
        do_step()

    true_hol = [e for e in fifo.hold_blocking_events if e['type'] == 'TRUE_HOL']
    arb_loss  = [e for e in fifo.hold_blocking_events if e['type'] == 'ARBITRATION_LOSS']

    print(f"\n{'=' * 70}")
    print(f"FIFO Results:")
    print(f"  Total Service Time      : {fifo.time} time slots")
    print(f"  True HoL Blocking Events: {len(true_hol)}")
    print(f"  Arbitration Loss Events : {len(arb_loss)}")

    return fifo


## Optimal VOQ Runner

In [28]:
def run_optimal_with_logging():
    packets = load_packets()
    opt     = OptimalVOQSwitch(3)

    print("\n" + "=" * 70)
    print("PART 2: OPTIMAL VOQ STEP-BY-STEP SIMULATION")
    print("=" * 70)

    max_arr = max(p.arrival_time for p in packets)

    def do_step():
        print(f"\n{'=' * 60}")
        print(f"TIME SLOT t = {opt.time}")
        print(f"{'=' * 60}")
        print("\nQueue BEFORE sending:")
        print(opt.get_queue_state_string())

        opt.step_optimal(record_log=True)
        last = opt.step_logs[-1]

        print("\nMatching Chosen:")
        for m in last['matching']:
            print(f"  → {m}")

        print("\nPackets SENT:")
        print(" ", last['sent_packets'])

        print(f"\nBacklog remaining after slot {last['time']}: {opt.backlog_history[-1]}")
        print("\nQueue AFTER sending:")
        print(opt.get_queue_state_string())

    for t in range(max_arr + 1):
        for p in packets:
            if p.arrival_time == t:
                opt.add_packet(p)
                print(f"\n[ARRIVAL] t={t}: {p.pid} (I{p.src_input} → O{p.dest_output})")
        if opt.get_backlog() > 0:
            do_step()
        else:
            print(f"\n[IDLE] t={opt.time}")
            opt.record_idle()

    while opt.get_backlog() > 0:
        do_step()

    print(f"\n{'=' * 70}")
    print(f"Optimal VOQ Results:")
    print(f"  Total Service Time: {opt.time} time slots")

    return opt

## iSLIP Runner

In [29]:
def run_islip_with_logging():
    packets = load_packets()
    islip   = iSLIPSwitch(3)

    print("\n" + "=" * 70)
    print("PART 3: iSLIP STEP-BY-STEP SIMULATION")
    print("=" * 70)
    print("\nInitial Pointers — Grant: [0,0,0]  Accept: [0,0,0]")

    max_arr = max(p.arrival_time for p in packets)

    def do_step():
        print(f"\n{'=' * 70}")
        print(f"TIME SLOT t = {islip.time}")
        print(f"{'=' * 70}")

        print("\n  Queue State BEFORE sending:")
        print(f"    {islip.get_queue_state_string()}")

        requests = islip.get_requests()
        print("\n  REQUEST PHASE:")
        for i in range(3):
            reqs = [f"O{j}" for j in range(3) if requests[i][j]]
            print(f"    I{i} requests: {', '.join(reqs) if reqs else 'none'}")

        islip.step_islip(record_log=True)
        last = islip.step_logs[-1]

        print("\n  iSLIP ITERATIONS:")
        for det in last['iteration_details']:
            print(f"\n    Iteration {det['iteration']}:")
            print("      GRANT PHASE:")
            for g in det['grant_details']:
                print(f"        {g}")
            print("      ACCEPT PHASE:")
            for a in det['accept_details']:
                print(f"        {a}")
            if det['matched'] != "No matches this iteration":
                print(f"      ✓ MATCH:  {det['matched']}")
                print(f"      → {det['ptr_updates']}")
            else:
                print(f"      ✗ {det['matched']}")

        print(f"\n  FINAL MATCHING:")
        for m in last['final_matches']:
            print(f"    ✓ {m}")

        print(f"\n  Packets SENT:")
        for pid in last['sent_packets']:
            print(f"    → {pid}")

        print(f"\n  Backlog remaining after slot {last['time']}: {islip.backlog_history[-1]}")

        print(f"\n  Updated Pointers:")
        print(f"    Grant  (O0,O1,O2): {last['grant_pointers']}")
        print(f"    Accept (I0,I1,I2): {last['accept_pointers']}")

        print(f"\n  Queue State AFTER sending:")
        print(f"    {islip.get_queue_state_string()}")

    for t in range(max_arr + 1):
        for p in packets:
            if p.arrival_time == t:
                islip.add_packet(p)
                print(f"\n[ARRIVAL] t={t}: {p.pid} (I{p.src_input} → O{p.dest_output})")
        if islip.get_backlog() > 0:
            do_step()
        else:
            print(f"\n[IDLE] t={islip.time}")
            islip.record_idle()

    while islip.get_backlog() > 0:
        do_step()

    print(f"\n{'=' * 70}")
    print(f"iSLIP Results:")
    print(f"  Total Service Time: {islip.time} time slots")

    return islip


# Visualization Functions

In [30]:
def visualize_results(fifo, opt, islip):
    fifo_time  = fifo.time
    opt_time   = opt.time
    islip_time = islip.time

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("Network Switch Scheduling Algorithm Comparison",
                 fontsize=15, fontweight='bold')

    # ── Graph 1: Service Time Bar Chart ──────────────────────────────
    algorithms = ['FIFO\n(HoL blocking)', 'Optimal VOQ\n(theoretical best)', 'iSLIP VOQ\n(practical)']
    times      = [fifo_time, opt_time, islip_time]
    colors     = ['#E74C3C', '#2ECC71', '#3498DB']

    bars = ax1.bar(algorithms, times, color=colors, edgecolor='black', linewidth=1.5, width=0.5)
    ax1.set_ylabel('Total Time Slots Required', fontsize=12, fontweight='bold')
    ax1.set_title('Graph 1: Total Service Time', fontsize=13, fontweight='bold')
    ax1.set_ylim(0, max(times) + 3)
    ax1.grid(axis='y', alpha=0.3, linestyle='--')

    for bar, t_val in zip(bars, times):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                 str(t_val), ha='center', va='bottom', fontweight='bold', fontsize=13)

    for idx, t_val in enumerate(times[1:], 1):
        pct = (1 - t_val / times[0]) * 100
        ax1.text(bars[idx].get_x() + bars[idx].get_width() / 2,
                 t_val / 2,
                 f'{pct:.0f}%\nfaster',
                 ha='center', va='center', fontweight='bold',
                 color='white', fontsize=10)

    # ── Graph 2: Backlog over Time ────────────────────────────────────
    fb = fifo.backlog_history
    ob = opt.backlog_history
    ib = islip.backlog_history

    max_t = max(len(fb), len(ob), len(ib))

    def pad(lst, n):
        return lst + [0] * (n - len(lst))

    fb_padded = pad(fb, max_t)
    ob_padded = pad(ob, max_t)
    ib_padded = pad(ib, max_t)
    xs = list(range(max_t))

    ax2.plot(xs, fb_padded, 'o-', color='#E74C3C', linewidth=2, markersize=6,
             label=f'FIFO (done @ end of t={fifo_time - 1})')
    ax2.plot(xs, ob_padded, 's-', color='#2ECC71', linewidth=2, markersize=6,
             label=f'Optimal VOQ (done @ end of t={opt_time - 1})')
    ax2.plot(xs, ib_padded, '^-', color='#3498DB', linewidth=2, markersize=6,
             label=f'iSLIP VOQ (done @ end of t={islip_time - 1})')

    ax2.axvline(x=5, color='gray', linestyle='--', alpha=0.6, linewidth=1.5)
    ax2.text(5.1, max(fb_padded) * 0.95, 'Last arrival\n(t=5)',
             fontsize=9, color='gray')

    ax2.set_xlabel('Time Slot (t)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Packets Remaining After Slot t', fontsize=12, fontweight='bold')
    ax2.set_title('Graph 2: Packet Backlog Over Time\n'
                  '(packets remaining AFTER each slot sends)',
                  fontsize=13, fontweight='bold')
    ax2.set_xticks(range(max_t))
    ax2.set_ylim(bottom=0)
    ax2.legend(loc='upper right', fontsize=10, framealpha=0.9)
    ax2.grid(True, alpha=0.3, linestyle='--')

    true_hol = len([e for e in fifo.hold_blocking_events if e['type'] == 'TRUE_HOL'])
    stats = (f'FIFO: {fifo_time} slots\n'
             f'Optimal: {opt_time} slots\n'
             f'iSLIP: {islip_time} slots\n'
             f'True HoL events: {true_hol}')
    ax2.text(0.02, 0.98, stats, transform=ax2.transAxes,
             fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout(pad=3.0)
    fpath = os.path.join(RESULTS_DIR, 'switch_simulation_results.png')
    plt.savefig(fpath, dpi=150, bbox_inches='tight')
    print(f"\n✓ Comparison graph saved → {fpath}")
    plt.close()
    
# Gantt chart

def plot_gantt(log, title):
    if not log:
        print(f"No data for Gantt: {title}")
        return

    time_slots = {}
    for entry in log:
        time_slots.setdefault(entry['time'], []).append(entry['packet_id'])

    fig, ax = plt.subplots(figsize=(12, max(6, len(log) * 0.4)))

    y_pos    = 0
    y_ticks  = []
    y_labels = []

    for t in sorted(time_slots):
        for pid in time_slots[t]:
            ax.barh(y_pos, 1, left=t, edgecolor='black', linewidth=0.5)
            ax.text(t + 0.5, y_pos, pid, ha='center', va='center', fontsize=9)
            y_ticks.append(y_pos)
            y_labels.append(pid)
            y_pos += 1.5
        y_pos += 0.5

    ax.set_xlabel("Time Slot", fontsize=12)
    ax.set_ylabel("Packets", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()

    fname = title.replace(' ', '_') + '.png'
    fpath = os.path.join(RESULTS_DIR, fname)
    plt.savefig(fpath, dpi=120, bbox_inches='tight')
    print(f"✓ Gantt chart saved  → {fpath}")
    plt.close()
    
    

# Metrics Functions

In [31]:
def compute_average_delay(packets, sent_log):
    arrival = {p.pid: p.arrival_time for p in packets}
    delays  = [entry['time'] - arrival[entry['packet_id']] for entry in sent_log]
    return sum(delays) / len(delays) if delays else 0


def compute_throughput(total_packets, total_time):
    return total_packets / total_time if total_time > 0 else 0


def print_service_time_comparison(fifo_time, opt_time, islip_time):
    print("\n" + "=" * 60)
    print("SERVICE TIME COMPARISON")
    print("=" * 60)
    print(f"{'Algorithm':<20} {'Time Slots':<15} {'vs FIFO'}")
    print("-" * 60)
    print(f"{'FIFO':<20} {fifo_time:<15} —")
    print(f"{'Optimal VOQ':<20} {opt_time:<15} {(1 - opt_time/fifo_time)*100:+.1f}%")
    print(f"{'iSLIP':<20} {islip_time:<15} {(1 - islip_time/fifo_time)*100:+.1f}%")

# Main Execution

In [33]:
fifo = run_fifo_with_logging()
opt = run_optimal_with_logging()
islip = run_islip_with_logging()

print_service_time_comparison(fifo.time, opt.time, islip.time)

packets = load_packets()
n = len(packets)

fifo_delay = compute_average_delay(packets, fifo.sent_packets_log)
islip_delay = compute_average_delay(packets, islip.sent_packets_log)

print("\nFINAL PERFORMANCE METRICS")
print("="*50)

print("\nFIFO:")
print(f"Average Delay : {fifo_delay:.2f}")
print(f"Throughput    : {compute_throughput(n, fifo.time):.3f} packets/slot")

print("\nOptimal VOQ:")
print(f"Throughput    : {compute_throughput(n, opt.time):.3f} packets/slot")

print("\niSLIP:")
print(f"Average Delay : {islip_delay:.2f}")
print(f"Throughput    : {compute_throughput(n, islip.time):.3f} packets/slot")

visualize_results(fifo, opt, islip)
plot_gantt(fifo.sent_packets_log, "FIFO_Packet_Timeline")
plot_gantt(islip.sent_packets_log, "iSLIP_Packet_Timeline")


PART 1: FIFO STEP-BY-STEP SIMULATION

[ARRIVAL] t=0: p1 (I0 → O0)

[ARRIVAL] t=0: p2 (I0 → O1)

[ARRIVAL] t=0: p3 (I1 → O0)

[ARRIVAL] t=0: p4 (I1 → O2)

[ARRIVAL] t=0: p5 (I2 → O0)

TIME SLOT t = 0

Queue BEFORE sending:
I0: [p1(→O0) | p2(→O1)]
        I1: [p3(→O0) | p4(→O2)]
        I2: [p5(→O0)]

Packets SENT:
  → p1

HoL / Arbitration Events:
  [TRUE_HOL] p4: TRUE HoL: p4 wants O2 (idle) but blocked by head p3 → O0
  [ARBITRATION_LOSS] p3: Lost arbitration to I0 for output O0
  [ARBITRATION_LOSS] p5: Lost arbitration to I0 for output O0

Backlog remaining after slot 0: 4

Queue AFTER sending:
I0: [p2(→O1)]
        I1: [p3(→O0) | p4(→O2)]
        I2: [p5(→O0)]

[ARRIVAL] t=1: p6 (I0 → O2)

[ARRIVAL] t=1: p7 (I2 → O1)

TIME SLOT t = 1

Queue BEFORE sending:
I0: [p2(→O1) | p6(→O2)]
        I1: [p3(→O0) | p4(→O2)]
        I2: [p5(→O0) | p7(→O1)]

Packets SENT:
  → p3
  → p2

HoL / Arbitration Events:
  [ARBITRATION_LOSS] p5: Lost arbitration to I1 for output O0

Backlog remaining afte